# Exemplo feito em sala pelo professor

In [ ]:
// Se o scanner ler: "$a = 10;"
// O tradutor escreve no arquivo .c:
fprintf(arquivo_saida, "float a = 10;\n");

// Se o scanner ler: "print($a);"
// O tradutor escreve:
fprintf(arquivo_saida, "printf(\"%%f\\n\", a);\n");

In [ ]:
import re

# 1. DEFINIÇÃO DO CÓDIGO FONTE "LUCAS"
codigo_lucas = """
$a = 10;
$b = 20;
$soma = $a + $b;
$media = $soma / 2;
print($media);
"""

def compilador_lucas_para_c(script):
    # 2. ANÁLISE LÉXICA (Mapeamento de Tokens via Regex)
    # Substituimos as variáveis $nome por nomes válidos em C
    # \$([a-z]+) -> procura o cifrão e captura o nome
    passo1 = re.sub(r'\$([a-z]+)', r'float \1', script)

    # Traduzimos o comando print(var) para printf do C
    # print\(float ([a-z]+)\) -> busca o padrão print e a variável já convertida
    passo2 = re.sub(r'print\(float ([a-z]+)\);', r'printf("%.2f\\n", \1);', passo1)
    # Traduzimos o comando IF para printf do C
    passo3 = re.sub(r'se\s*\((.*)\)\s*\{', r'if (\1) {', passo2)

    # 3. GERAÇÃO DE ESTRUTURA (Boilerplate de C)
    codigo_c = f"""#include <stdio.h>

int main() {{
    {passo2.strip()}
    return 0;
}}
"""
    return codigo_c

# Executando a Transpilação
resultado_c = compilador_lucas_para_c(codigo_lucas)

print("--- CÓDIGO GERADO EM C ---")
print(resultado_c)

# 4. SALVANDO O ARQUIVO (Opcional)
with open("programa_gerado.c", "w") as f:
    f.write(resultado_c)

In [ ]:
import re

# 1. CÓDIGO FONTE NA LINGUAGEM "LUCAS"
# Vamos forçar um erro semântico comentando a linha do $b
codigo_lucas = """
$a = 10;
$b = 20;
$soma = $a + $b;
$media = $soma / 2;
print($media);
print($soma);
"""

class CompiladorLucas:
    def __init__(self):
        self.tabela_simbolos = {} # Nome da variável -> Tipo/Valor

    def analisar_lexico_e_semantico(self, script):
        print("--- INICIANDO ANÁLISE ---")
        linhas = script.strip().split('\n')
        codigo_c_corpo = ""

        for i, linha in enumerate(linhas):
            linha = linha.strip()
            if not linha: continue

            # REGEX PARA ATRIBUIÇÃO: $variavel = valor;
            match_atrib = re.match(r'\$([a-z]+)\s*=\s*(.*);', linha)
            if match_atrib:
                var_nome = match_atrib.group(1)
                expressao = match_atrib.group(2)

                # SEMÂNTICA: Verificar se as variáveis na expressão existem
                vars_na_expressao = re.findall(r'\$([a-z]+)', expressao)
                for v in vars_na_expressao:
                    if v not in self.tabela_simbolos:
                        raise NameError(f"Erro Semântico na linha {i+1}: Variável '${v}' não declarada!")

                # Registrar na Tabela de Símbolos
                self.tabela_simbolos[var_nome] = "float"

                # Tradução para C (Lidando com a primeira declaração)
                expressao_limpa = expressao.replace('$', '')
                codigo_c_corpo += f"    float {var_nome} = {expressao_limpa};\n"

            # REGEX PARA PRINT: print($var);
            match_print = re.match(r'print\(\$([a-z]+)\);', linha)
            if match_print:
                var_nome = match_print.group(1)
                if var_nome not in self.tabela_simbolos:
                    raise NameError(f"Erro Semântico: Tentativa de imprimir '${var_nome}' não declarada.")

                codigo_c_corpo += f'    printf("%.2f\\n", {var_nome});\n'

        return codigo_c_corpo

    def gerar_codigo_final(self, corpo):
        return f"""#include <stdio.h>

int main() {{
{corpo}
    return 0;
}}"""

# --- EXECUÇÃO NO JUPYTER ---
try:
    c_lucas = CompiladorLucas()
    corpo_traduzido = c_lucas.analisar_lexico_e_semantico(codigo_lucas)
    codigo_final = c_lucas.gerar_codigo_final(corpo_traduzido)

    print("\n[SUCESSO] Tabela de Símbolos final:", c_lucas.tabela_simbolos)
    print("\n--- CÓDIGO C GERADO ---")
    print(codigo_final)

except Exception as e:
    print(f"\n[ERRO DO COMPILADOR] {e}")

--- INICIANDO ANÁLISE ---

[SUCESSO] Tabela de Símbolos final: {'a': 'float', 'b': 'float', 'soma': 'float', 'media': 'float'}

--- CÓDIGO C GERADO ---
#include <stdio.h>

int main() {
    float a = 10;
    float b = 20;
    float soma = a + b;
    float media = soma / 2;
    printf("%.2f\n", media);
    printf("%.2f\n", soma);

    return 0;
}


# Análise Léxica Dotmon para C

Documentação da linguagem dotmon

https://docs.google.com/document/d/1M5pu2B9n9eChZ0szS1Pfo6Adb-2Wuj0KlmNeCBuvjNw/edit?pli=1&tab=t.0#heading=h.9a3guzsfz6y8

## Scanner Dotmon para C

```
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <stdbool.h>

typedef enum {
    // Tipos
    TK_TYPE_BABY,
    TK_TYPE_PUP,
    TK_TYPE_ROOK,
    TK_TYPE_CHAMP,
    TK_TYPE_MOJI,
    TK_TYPE_BIT,

    // Controle de fluxo
    TK_KW_EVO,
    TK_KW_FAILEVO,
    TK_KW_ALTEVO,
    TK_KW_JAM,
    TK_KW_SKIP,

    // Funções
    TK_KW_XROS,
    TK_KW_SEND,

    // Loops
    TK_KW_LOOP,
    TK_KW_SPIRAL,

    // Interoperabilidade
    TK_KW_WORLD,
    TK_KW_CORE,
    TK_KW_CALL,

    // Entrada / saída
    TK_KW_SHOW,
    TK_KW_ASK,

    // Programa
    TK_KW_START,
    TK_KW_FINISH,

    // Identificadores e literais
    TK_IDENTIFIER,
    TK_INT_LITERAL,
    TK_FLOAT_LITERAL,
    TK_BOOL_LITERAL,
    TK_STRING_LITERAL,
    TK_CHAR_LITERAL,

    // Operadores
    TK_OP_ASSIGN,   // =
    TK_OP_PLUS,     // +
    TK_OP_MINUS,    // -
    TK_OP_MUL,      // *
    TK_OP_DIV,      // /
    TK_OP_EQ,       // ==
    TK_OP_NE,       // !=
    TK_OP_GT,       // >
    TK_OP_LT,       // <
    TK_OP_GE,       // >=
    TK_OP_LE,       // <=

    // Pontuação
    TK_LPAREN,      // (
    TK_RPAREN,      // )
    TK_LBRACE,      // {
    TK_RBRACE,      // }
    TK_SEMICOLON,   // ;
    TK_COMMA,       // ,
    TK_DOT,         // .

    // Controle interno
    TK_EOF,
    TK_INVALID
} TokenType;

typedef struct {
    TokenType type;
    char *lexeme;
    int line;
    int column;
} Token;

typedef struct {
    const char *source;
    size_t length;
    size_t start;
    size_t current;
    int line;
    int column;
    int token_line;
    int token_column;
} Scanner;

typedef struct {
    const char *keyword;
    TokenType type;
} KeywordEntry;

static const KeywordEntry RESERVED[] = {
    {"Baby",   TK_TYPE_BABY},
    {"Pup",    TK_TYPE_PUP},
    {"Rook",   TK_TYPE_ROOK},
    {"Champ",  TK_TYPE_CHAMP},
    {"Moji",   TK_TYPE_MOJI},
    {"Bit",    TK_TYPE_BIT},

    {"Evo",     TK_KW_EVO},
    {"FailEvo", TK_KW_FAILEVO},
    {"AltEvo",  TK_KW_ALTEVO},
    {"Jam",     TK_KW_JAM},
    {"Skip",    TK_KW_SKIP},

    {"Xros", TK_KW_XROS},
    {"Send", TK_KW_SEND},

    {"Loop",   TK_KW_LOOP},
    {"Spiral", TK_KW_SPIRAL},

    {"World", TK_KW_WORLD},
    {"Core",  TK_KW_CORE},
    {"Call",  TK_KW_CALL},

    {"Show", TK_KW_SHOW},
    {"Ask",  TK_KW_ASK},

    {"Start",  TK_KW_START},
    {"Finish", TK_KW_FINISH},

    {"true",  TK_BOOL_LITERAL},
    {"false", TK_BOOL_LITERAL}
};

static const size_t RESERVED_COUNT = sizeof(RESERVED) / sizeof(RESERVED[0]);

static void scanner_init(Scanner *scanner, const char *source) {
    scanner->source = source;
    scanner->length = strlen(source);
    scanner->start = 0;
    scanner->current = 0;
    scanner->line = 1;
    scanner->column = 1;
    scanner->token_line = 1;
    scanner->token_column = 1;
}

static bool is_at_end(const Scanner *scanner) {
    return scanner->current >= scanner->length;
}

static char peek(const Scanner *scanner) {
    if (is_at_end(scanner)) return '\0';
    return scanner->source[scanner->current];
}

static char peek_next(const Scanner *scanner) {
    if (scanner->current + 1 >= scanner->length) return '\0';
    return scanner->source[scanner->current + 1];
}

static char advance_char(Scanner *scanner) {
    char c = scanner->source[scanner->current++];
    if (c == '\n') {
        scanner->line++;
        scanner->column = 1;
    } else {
        scanner->column++;
    }
    return c;
}

static bool match_char(Scanner *scanner, char expected) {
    if (is_at_end(scanner)) return false;
    if (scanner->source[scanner->current] != expected) return false;
    advance_char(scanner);
    return true;
}

static char *copy_lexeme(const Scanner *scanner) {
    size_t len = scanner->current - scanner->start;
    char *text = (char *)malloc(len + 1);
    if (text == NULL) {
        fprintf(stderr, "Erro de memória ao copiar lexema.\n");
        exit(EXIT_FAILURE);
    }
    memcpy(text, scanner->source + scanner->start, len);
    text[len] = '\0';
    return text;
}

static Token make_token(const Scanner *scanner, TokenType type) {
    Token token;
    token.type = type;
    token.lexeme = copy_lexeme(scanner);
    token.line = scanner->token_line;
    token.column = scanner->token_column;
    return token;
}

static Token make_invalid_token(const Scanner *scanner, const char *message) {
    Token token;
    token.type = TK_INVALID;
    token.lexeme = (char *)malloc(strlen(message) + 1);
    if (token.lexeme == NULL) {
        fprintf(stderr, "Erro de memória ao criar token inválido.\n");
        exit(EXIT_FAILURE);
    }
    strcpy(token.lexeme, message);
    token.line = scanner->token_line;
    token.column = scanner->token_column;
    return token;
}

static bool is_identifier_start(char c) {
    return isalpha((unsigned char)c) || c == '_';
}

static bool is_identifier_part(char c) {
    return isalnum((unsigned char)c) || c == '_';
}

static void skip_whitespace_and_comments(Scanner *scanner) {
    for (;;) {
        char c = peek(scanner);

        if (c == ' ' || c == '\r' || c == '\t' || c == '\n') {
            advance_char(scanner);
            continue;
        }

        if (c == '/' && peek_next(scanner) == '/') {
            advance_char(scanner); // /
            advance_char(scanner); // /
            while (!is_at_end(scanner) && peek(scanner) != '\n') {
                advance_char(scanner);
            }
            continue;
        }

        if (c == '/' && peek_next(scanner) == '*') {
            advance_char(scanner); // /
            advance_char(scanner); // *
            while (!is_at_end(scanner)) {
                if (peek(scanner) == '*' && peek_next(scanner) == '/') {
                    advance_char(scanner); // *
                    advance_char(scanner); // /
                    break;
                }
                advance_char(scanner);
            }
            continue;
        }

        break;
    }
}

static TokenType identifier_type(const Scanner *scanner) {
    size_t len = scanner->current - scanner->start;
    const char *text = scanner->source + scanner->start;

    for (size_t i = 0; i < RESERVED_COUNT; i++) {
        if (strlen(RESERVED[i].keyword) == len &&
            strncmp(text, RESERVED[i].keyword, len) == 0) {
            return RESERVED[i].type;
        }
    }

    return TK_IDENTIFIER;
}

static Token scan_identifier(Scanner *scanner) {
    while (is_identifier_part(peek(scanner))) {
        advance_char(scanner);
    }
    return make_token(scanner, identifier_type(scanner));
}

static Token scan_number(Scanner *scanner) {
    while (isdigit((unsigned char)peek(scanner))) {
        advance_char(scanner);
    }

    if (peek(scanner) == '.' && isdigit((unsigned char)peek_next(scanner))) {
        advance_char(scanner); // .
        while (isdigit((unsigned char)peek(scanner))) {
            advance_char(scanner);
        }
        return make_token(scanner, TK_FLOAT_LITERAL);
    }

    return make_token(scanner, TK_INT_LITERAL);
}

static Token scan_string(Scanner *scanner) {
    while (!is_at_end(scanner) && peek(scanner) != '"') {
        if (peek(scanner) == '\n') {
            return make_invalid_token(scanner, "String nao encerrada.");
        }
        advance_char(scanner);
    }

    if (is_at_end(scanner)) {
        return make_invalid_token(scanner, "String nao encerrada.");
    }

    advance_char(scanner); // fecha "
    return make_token(scanner, TK_STRING_LITERAL);
}

static Token scan_char_literal(Scanner *scanner) {
    if (is_at_end(scanner) || peek(scanner) == '\n' || peek(scanner) == '\'') {
        return make_invalid_token(scanner, "Literal char invalido.");
    }

    advance_char(scanner); // caractere interno

    if (!match_char(scanner, '\'')) {
        return make_invalid_token(scanner, "Literal char invalido.");
    }

    return make_token(scanner, TK_CHAR_LITERAL);
}

static Token scan_token(Scanner *scanner) {
    skip_whitespace_and_comments(scanner);

    scanner->start = scanner->current;
    scanner->token_line = scanner->line;
    scanner->token_column = scanner->column;

    if (is_at_end(scanner)) {
        return make_token(scanner, TK_EOF);
    }

    char c = advance_char(scanner);

    if (is_identifier_start(c)) {
        return scan_identifier(scanner);
    }

    if (isdigit((unsigned char)c)) {
        return scan_number(scanner);
    }

    switch (c) {
        case '"':
            return scan_string(scanner);

        case '\'':
            return scan_char_literal(scanner);

        case '(':
            return make_token(scanner, TK_LPAREN);
        case ')':
            return make_token(scanner, TK_RPAREN);
        case '{':
            return make_token(scanner, TK_LBRACE);
        case '}':
            return make_token(scanner, TK_RBRACE);
        case ';':
            return make_token(scanner, TK_SEMICOLON);
        case ',':
            return make_token(scanner, TK_COMMA);
        case '.':
            return make_token(scanner, TK_DOT);

        case '+':
            return make_token(scanner, TK_OP_PLUS);
        case '-':
            return make_token(scanner, TK_OP_MINUS);
        case '*':
            return make_token(scanner, TK_OP_MUL);
        case '/':
            return make_token(scanner, TK_OP_DIV);

        case '=':
            if (match_char(scanner, '=')) return make_token(scanner, TK_OP_EQ);
            return make_token(scanner, TK_OP_ASSIGN);

        case '!':
            if (match_char(scanner, '=')) return make_token(scanner, TK_OP_NE);
            return make_invalid_token(scanner, "Token invalido: '!' isolado.");

        case '>':
            if (match_char(scanner, '=')) return make_token(scanner, TK_OP_GE);
            return make_token(scanner, TK_OP_GT);

        case '<':
            if (match_char(scanner, '=')) return make_token(scanner, TK_OP_LE);
            return make_token(scanner, TK_OP_LT);

        default:
            return make_invalid_token(scanner, "Caractere invalido.");
    }
}

static const char *token_type_name(TokenType type) {
    switch (type) {
        case TK_TYPE_BABY: return "TYPE_BABY";
        case TK_TYPE_PUP: return "TYPE_PUP";
        case TK_TYPE_ROOK: return "TYPE_ROOK";
        case TK_TYPE_CHAMP: return "TYPE_CHAMP";
        case TK_TYPE_MOJI: return "TYPE_MOJI";
        case TK_TYPE_BIT: return "TYPE_BIT";

        case TK_KW_EVO: return "KW_EVO";
        case TK_KW_FAILEVO: return "KW_FAILEVO";
        case TK_KW_ALTEVO: return "KW_ALTEVO";
        case TK_KW_JAM: return "KW_JAM";
        case TK_KW_SKIP: return "KW_SKIP";

        case TK_KW_XROS: return "KW_XROS";
        case TK_KW_SEND: return "KW_SEND";

        case TK_KW_LOOP: return "KW_LOOP";
        case TK_KW_SPIRAL: return "KW_SPIRAL";

        case TK_KW_WORLD: return "KW_WORLD";
        case TK_KW_CORE: return "KW_CORE";
        case TK_KW_CALL: return "KW_CALL";

        case TK_KW_SHOW: return "KW_SHOW";
        case TK_KW_ASK: return "KW_ASK";

        case TK_KW_START: return "KW_START";
        case TK_KW_FINISH: return "KW_FINISH";

        case TK_IDENTIFIER: return "IDENTIFIER";
        case TK_INT_LITERAL: return "INT_LITERAL";
        case TK_FLOAT_LITERAL: return "FLOAT_LITERAL";
        case TK_BOOL_LITERAL: return "BOOL_LITERAL";
        case TK_STRING_LITERAL: return "STRING_LITERAL";
        case TK_CHAR_LITERAL: return "CHAR_LITERAL";

        case TK_OP_ASSIGN: return "OP_ASSIGN";
        case TK_OP_PLUS: return "OP_PLUS";
        case TK_OP_MINUS: return "OP_MINUS";
        case TK_OP_MUL: return "OP_MUL";
        case TK_OP_DIV: return "OP_DIV";
        case TK_OP_EQ: return "OP_EQ";
        case TK_OP_NE: return "OP_NE";
        case TK_OP_GT: return "OP_GT";
        case TK_OP_LT: return "OP_LT";
        case TK_OP_GE: return "OP_GE";
        case TK_OP_LE: return "OP_LE";

        case TK_LPAREN: return "LPAREN";
        case TK_RPAREN: return "RPAREN";
        case TK_LBRACE: return "LBRACE";
        case TK_RBRACE: return "RBRACE";
        case TK_SEMICOLON: return "SEMICOLON";
        case TK_COMMA: return "COMMA";
        case TK_DOT: return "DOT";

        case TK_EOF: return "EOF";
        case TK_INVALID: return "INVALID";
        default: return "UNKNOWN";
    }
}

static void print_token(const Token *token) {
    printf("[%d:%d] %-15s -> %s\n",
           token->line,
           token->column,
           token_type_name(token->type),
           token->lexeme);
}

static void free_token(Token *token) {
    free(token->lexeme);
    token->lexeme = NULL;
}

static char *read_file(const char *filename) {
    FILE *file = fopen(filename, "rb");
    if (file == NULL) {
        fprintf(stderr, "Nao foi possivel abrir o arquivo: %s\n", filename);
        return NULL;
    }

    if (fseek(file, 0, SEEK_END) != 0) {
        fclose(file);
        fprintf(stderr, "Erro ao posicionar no fim do arquivo.\n");
        return NULL;
    }

    long size = ftell(file);
    if (size < 0) {
        fclose(file);
        fprintf(stderr, "Erro ao obter tamanho do arquivo.\n");
        return NULL;
    }

    rewind(file);

    char *buffer = (char *)malloc((size_t)size + 1);
    if (buffer == NULL) {
        fclose(file);
        fprintf(stderr, "Erro de memoria ao ler arquivo.\n");
        return NULL;
    }

    size_t bytes_read = fread(buffer, 1, (size_t)size, file);
    buffer[bytes_read] = '\0';
    fclose(file);

    return buffer;
}

int main(int argc, char *argv[]) {
    if (argc != 2) {
        fprintf(stderr, "Uso: %s <arquivo.dot>\n", argv[0]);
        return EXIT_FAILURE;
    }

    char *source = read_file(argv[1]);
    if (source == NULL) {
        return EXIT_FAILURE;
    }

    Scanner scanner;
    scanner_init(&scanner, source);

    for (;;) {
        Token token = scan_token(&scanner);
        print_token(&token);

        if (token.type == TK_EOF) {
            free_token(&token);
            break;
        }

        free_token(&token);
    }

    free(source);
    return EXIT_SUCCESS;
}
```

Código fonte feito em Dotmon

```
Start {
    Baby nivel = 10;
    Moji nome = "Agumon";

    Evo (nivel > 10) {
        Show("Nivel alto");
    } AltEvo (nivel == 10) {
        Show(nome);
    } FailEvo {
        Show("Nivel baixo");
    }
} Finish
```



Árvore hierárquica (GPT)


```
Programa
├── InicioPrograma: Start
├── BlocoPrincipal
│   ├── DeclaracaoVariavel
│   │   ├── Tipo: Baby
│   │   ├── Identificador: nivel
│   │   └── ValorInicial
│   │       └── Inteiro: 10
│   │
│   ├── DeclaracaoVariavel
│   │   ├── Tipo: Moji
│   │   ├── Identificador: nome
│   │   └── ValorInicial
│   │       └── String: "Agumon"
│   │
│   └── EstruturaCondicional
│       ├── If: Evo
│       │   ├── Condicao
│       │   │   ├── Identificador: nivel
│       │   │   ├── Operador: >
│       │   │   └── Inteiro: 10
│       │   └── Bloco
│       │       └── ComandoSaida
│       │           └── Show("Nivel alto")
│       │
│       ├── ElseIf: AltEvo
│       │   ├── Condicao
│       │   │   ├── Identificador: nivel
│       │   │   ├── Operador: ==
│       │   │   └── Inteiro: 10
│       │   └── Bloco
│       │       └── ComandoSaida
│       │           └── Show(nome)
│       │
│       └── Else: FailEvo
│           └── Bloco
│               └── ComandoSaida
│                   └── Show("Nivel baixo")
└── FimPrograma: Finish
```



Árvore Hierárquica no formato AST (GPT)



```
Program
├── VarDecl(type=Baby, name=nivel, value=10)
├── VarDecl(type=Moji, name=nome, value="Agumon")
└── IfStmt
    ├── condition: BinaryExpr(>, Identifier(nivel), Int(10))
    ├── thenBranch:
    │   └── ShowStmt(String("Nivel alto"))
    ├── elseIfBranch:
    │   ├── condition: BinaryExpr(==, Identifier(nivel), Int(10))
    │   └── ShowStmt(Identifier(nome))
    └── elseBranch:
        └── ShowStmt(String("Nivel baixo"))
```



Árvore hierárquica (CLAUDE)

```
PROGRAM
├── KW: Start
├── BLOCK
│   ├── DECL_STMT
│   │   ├── KW_TYPE:    Baby
│   │   ├── IDENT:      nivel
│   │   ├── OP:         =
│   │   ├── EXPR
│   │   │   └── INT_LITERAL:  10
│   │   └── PUNCT:      ;
│   ├── DECL_STMT
│   │   ├── KW_TYPE:    Moji
│   │   ├── IDENT:      nome
│   │   ├── OP:         =
│   │   ├── EXPR
│   │   │   └── STR_LITERAL:  "Agumon"
│   │   └── PUNCT:      ;
│   └── COND_STMT
│       ├── EVO_CLAUSE
│       │   ├── KW:     Evo
│       │   ├── COND_EXPR
│       │   │   ├── IDENT:    nivel
│       │   │   ├── RELOP:    >
│       │   │   └── INT_LIT:  10
│       │   └── BLOCK
│       │       └── CALL_STMT
│       │           ├── FUNC_ID:  Show
│       │           ├── ARG_LIST
│       │           │   └── STR_LIT:  "Nivel alto"
│       │           └── PUNCT:  ;
│       ├── ALTEVO_CLAUSE
│       │   ├── KW:     AltEvo
│       │   ├── COND_EXPR
│       │   │   ├── IDENT:    nivel
│       │   │   ├── RELOP:    ==
│       │   │   └── INT_LIT:  10
│       │   └── BLOCK
│       │       └── CALL_STMT
│       │           ├── FUNC_ID:  Show
│       │           ├── ARG_LIST
│       │           │   └── IDENT:  nome
│       │           └── PUNCT:  ;
│       └── FAILEVO_CLAUSE
│           ├── KW:     FailEvo
│           └── BLOCK
│               └── CALL_STMT
│                   ├── FUNC_ID:  Show
│                   ├── ARG_LIST
│                   │   └── STR_LIT:  "Nivel baixo"
│                   └── PUNCT:  ;
└── KW: Finish
```

Árvore Hierárquica no formato AST (CLAUDE)

```
PROGRAM
└── BLOCK
    ├── VAR_DECL [Baby → Int]
    │   ├── name: nivel
    │   └── init: IntLiteral(10)
    ├── VAR_DECL [Moji → String]
    │   ├── name: nome
    │   └── init: StrLiteral("Agumon")
    └── IF_CHAIN
        ├── IF_BRANCH
        │   ├── test
        │   │   └── BinaryExpr
        │   │       ├── left:   Identifier(nivel)
        │   │       ├── op:     >
        │   │       └── right:  IntLiteral(10)
        │   └── body
        │       └── CallExpr
        │           ├── callee: Show
        │           └── args
        │               └── StrLiteral("Nivel alto")
        ├── ELSEIF_BRANCH
        │   ├── test
        │   │   └── BinaryExpr
        │   │       ├── left:   Identifier(nivel)
        │   │       ├── op:     ==
        │   │       └── right:  IntLiteral(10)
        │   └── body
        │       └── CallExpr
        │           ├── callee: Show
        │           └── args
        │               └── Identifier(nome)
        └── ELSE_BRANCH
            └── body
                └── CallExpr
                    ├── callee: Show
                    └── args
                        └── StrLiteral("Nivel baixo")
```



Saída correta do scanner

```
KW_START
LBRACE

TYPE_BABY
IDENTIFIER("nivel")
OP_ASSIGN
INT_LITERAL("10")
SEMICOLON

TYPE_MOJI
IDENTIFIER("nome")
OP_ASSIGN
STRING_LITERAL("\"Agumon\"")
SEMICOLON

KW_EVO
LPAREN
IDENTIFIER("nivel")
OP_GT
INT_LITERAL("10")
RPAREN
LBRACE
KW_SHOW
LPAREN
STRING_LITERAL("\"Nivel alto\"")
RPAREN
SEMICOLON
RBRACE

KW_ALTEVO
LPAREN
IDENTIFIER("nivel")
OP_EQ
INT_LITERAL("10")
RPAREN
LBRACE
KW_SHOW
LPAREN
IDENTIFIER("nome")
RPAREN
SEMICOLON
RBRACE

KW_FAILEVO
LBRACE
KW_SHOW
LPAREN
STRING_LITERAL("\"Nivel baixo\"")
RPAREN
SEMICOLON
RBRACE

RBRACE
KW_FINISH
EOF
```

